# Complex Engineering Problem (CEP)
**Course:** EE-475 Introduction to Machine Learning  
**Project:** Dual-View Multi-Task Breast Assessment using Swin Transformer  

---

## 1. Data Collection and Exploration (5 Marks)
**Objective:** We utilize the **VinDr-Mammo** dataset for breast cancer detection. Each patient case contains two complementary mammographic views:
* **CC (Cranio-Caudal):** Top-down view of the breast.
* **MLO (Medio-Lateral Oblique):** Angled side view of the breast.

**Dataset Details:**
* **Source:** VinDr-Mammo (Via Kaggle)
* **Input:** Paired CC + MLO images per study (Dual-View)
* **Target Variables (Multi-Task):**
  * **Task 1:** BI-RADS Assessment (5 classes: BI-RADS 1–5)
  * **Task 2:** Breast Density Classification (4 classes: Density A–D)
* **Imbalance Handling:** Class-Weighted Loss + WeightedRandomSampler

In this section, we locate the dataset files and perform an initial statistical summary.


# Step 1: Data Collection

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1.1 Locate Dataset Paths ---
base_path = '/kaggle/input'
IMG_DIR = None
CSV_PATH = None

print("🔍 Locating Dataset...")
for root, dirs, files in os.walk(base_path):
    if 'images_png' in dirs:
        IMG_DIR = os.path.join(root, 'images_png')
    elif 'processed_images' in dirs:
        IMG_DIR = os.path.join(root, 'processed_images')
    for file in files:
        if file == 'finding_annotations.csv':
            CSV_PATH = os.path.join(root, file)

if IMG_DIR and CSV_PATH:
    print(f"✅ Images Found: {IMG_DIR}")
    print(f"✅ Metadata Found: {CSV_PATH}")
else:
    raise FileNotFoundError("❌ Dataset not fully found. Please check input datasets.")

# --- 1.2 Data Exploration ---
df = pd.read_csv(CSV_PATH)

print(f"\n--- Dataset Summary ---")
print(f"Total Entries: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

print("\n--- Data Preview ---")
display(df.head())

print("\n--- Statistical Description ---")
print(df.describe())

print("\n--- Missing Values ---")
print(df.isnull().sum())

# --- 1.3 Target Distribution Overview ---
print("\n--- BI-RADS Assessment Distribution ---")
print(df['breast_birads'].value_counts().sort_index())

print("\n--- Breast Density Distribution ---")
print(df['breast_density'].value_counts().sort_index())


## 2. Data Preprocessing and Visualization

**Objective:** Prepare paired CC+MLO data for the Dual-View Multi-Task Swin Transformer.

**Steps Performed:**
1. **Data Cleaning:** Filter rows with missing images; drop rows missing `breast_density` or `breast_birads`.
2. **Dual-View Pairing:** Group by `study_id` and pair CC and MLO images. Studies missing either view are discarded.
3. **Label Encoding:**
   * **Task 1 — BI-RADS:** BI-RADS 1–5 → class index 0–4.
   * **Task 2 — Breast Density:** Density A–D → class index 0–3.
4. **Imbalance Handling:**
   * **Class-Weighted Loss:** Automatically compute inverse-frequency weights per BI-RADS class.
   * **WeightedRandomSampler:** Oversample minority classes at the DataLoader level.
5. **Preprocessing Pipeline:** Resize to 224×224, normalize with ImageNet statistics.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torchvision import transforms
from PIL import Image
import numpy as np

print("--- 2. Data Preprocessing & Visualization (Dual-View) ---")

# --- 2.1 Robust Load ---
if 'df' not in globals():
    print("⚠️ 'df' variable missing. Reloading CSV...")
    base_path = '/kaggle/input'
    CSV_PATH = None
    IMG_DIR = None
    for root, dirs, files in os.walk(base_path):
        if 'finding_annotations.csv' in files:
            CSV_PATH = os.path.join(root, 'finding_annotations.csv')
        if 'images_png' in dirs:
            IMG_DIR = os.path.join(root, 'images_png')
        elif 'processed_images' in dirs:
            IMG_DIR = os.path.join(root, 'processed_images')
    if CSV_PATH and IMG_DIR:
        df = pd.read_csv(CSV_PATH)
        print("✅ Data Reloaded Successfully.")
    else:
        raise FileNotFoundError("❌ Could not reload data. Please re-run Step 1.")

# --- 2.2 Label Encoding ---
BIRADS_MAP  = {'BI-RADS 1': 0, 'BI-RADS 2': 1, 'BI-RADS 3': 2, 'BI-RADS 4': 3, 'BI-RADS 5': 4}
DENSITY_MAP = {'DENSITY A': 0, 'DENSITY B': 1, 'DENSITY C': 2, 'DENSITY D': 3}

df['birads_label']  = df['breast_birads'].map(BIRADS_MAP)
df['density_label'] = df['breast_density'].map(DENSITY_MAP)
df = df.dropna(subset=['birads_label', 'density_label']).reset_index(drop=True)
df['birads_label']  = df['birads_label'].astype(int)
df['density_label'] = df['density_label'].astype(int)

# --- 2.3 Verify Image Existence ---
print("🔍 Verifying image files...")
def get_img_path(row, img_dir):
    img_name = str(row['image_id']) + '.png'
    p1 = os.path.join(img_dir, img_name)
    p2 = os.path.join(img_dir, str(row['study_id']), img_name)
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

df['img_path'] = df.apply(lambda r: get_img_path(r, IMG_DIR), axis=1)
df = df.dropna(subset=['img_path']).reset_index(drop=True)
print(f"Valid image rows: {len(df)}")

# --- 2.4 Dual-View Pairing ---
# Each study_id has multiple rows (CC and MLO per laterality).
# We pair CC and MLO within the same study_id + laterality.
print("\n🔗 Building Dual-View pairs (CC + MLO per study)...")

pairs = []
# Group by study_id and laterality
group_cols = ['study_id', 'laterality'] if 'laterality' in df.columns else ['study_id']
for keys, grp in df.groupby(group_cols):
    cc_rows  = grp[grp['view_position'] == 'CC']
    mlo_rows = grp[grp['view_position'] == 'MLO']
    if len(cc_rows) == 0 or len(mlo_rows) == 0:
        continue  # Skip if either view is missing
    # Take the first CC and first MLO (one pair per group)
    cc  = cc_rows.iloc[0]
    mlo = mlo_rows.iloc[0]
    pairs.append({
        'cc_path':       cc['img_path'],
        'mlo_path':      mlo['img_path'],
        'birads_label':  cc['birads_label'],   # Label from CC row (same study)
        'density_label': cc['density_label'],
        'breast_birads': cc['breast_birads'],
        'breast_density': cc['breast_density'],
    })

df_pairs = pd.DataFrame(pairs).reset_index(drop=True)
print(f"✅ Total Dual-View pairs: {len(df_pairs)}")

# --- 2.5 Visualizations ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    x='breast_birads', data=df_pairs, palette='Reds',
    order=['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5'], ax=axes[0]
)
axes[0].set_title('Task 1: BI-RADS Distribution (Paired Studies)')
axes[0].set_xlabel('BI-RADS Category')
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(
    x='breast_density', data=df_pairs, palette='Blues',
    order=['DENSITY A','DENSITY B','DENSITY C','DENSITY D'], ax=axes[1]
)
axes[1].set_title('Task 2: Breast Density Distribution (Paired Studies)')
axes[1].set_xlabel('Density Category')

plt.tight_layout()
plt.show()

# --- 2.6 Dual-View Multi-Task Dataset Class ---
class DualViewMammoDataset(Dataset):
    """
    Returns:
      img_cc      : Tensor (3, 224, 224) — CC view
      img_mlo     : Tensor (3, 224, 224) — MLO view
      label_birads  : int 0-4
      label_density : int 0-3
    """
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_cc  = Image.open(row['cc_path']).convert('RGB')
        img_mlo = Image.open(row['mlo_path']).convert('RGB')

        if self.transform:
            img_cc  = self.transform(img_cc)
            img_mlo = self.transform(img_mlo)

        lbl_birads  = torch.tensor(int(row['birads_label']),  dtype=torch.long)
        lbl_density = torch.tensor(int(row['density_label']), dtype=torch.long)

        return img_cc, img_mlo, lbl_birads, lbl_density

print("✅ DualViewMammoDataset class defined.")

# --- 2.7 Compute Class Weights for BI-RADS (for weighted loss & sampler) ---
birads_counts  = np.bincount(df_pairs['birads_label'].values, minlength=5)
density_counts = np.bincount(df_pairs['density_label'].values, minlength=4)

# Inverse frequency weights (normalized)
birads_weights  = 1.0 / (birads_counts + 1e-6)
birads_weights  = birads_weights / birads_weights.sum() * len(birads_weights)
density_weights = 1.0 / (density_counts + 1e-6)
density_weights = density_weights / density_weights.sum() * len(density_weights)

print("\n--- Class Weights ---")
birads_names  = ['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5']
density_names = ['Density A','Density B','Density C','Density D']
for name, w in zip(birads_names, birads_weights):
    print(f"  {name}: weight = {w:.4f}")

# Tensor weights for CrossEntropyLoss
birads_weight_tensor  = torch.tensor(birads_weights,  dtype=torch.float32)
density_weight_tensor = torch.tensor(density_weights, dtype=torch.float32)

print("✅ Class weights computed.")


## 3. Feature Extraction and Selection (5 Marks)
**Objective:** We implement a **Dual-View Swin Transformer (Swin-T)** that processes CC and MLO images through a **shared-weight backbone** and fuses their features before classification.

* **Backbone:** Swin-T pretrained on ImageNet — applied independently to CC and MLO images (shared weights).
* **Fusion:** Features from both views are concatenated → (B, 1536) → passed to classification heads.
* **Why shared weights?** Forces the backbone to learn view-agnostic mammographic features, reduces overfitting, and halves the number of backbone parameters.

## 4. Model Building and Training (5 Marks)
**Objective:** Train the `DualViewSwinModel` with multi-task learning and imbalance handling.

* **Algorithm:** Dual-View Swin-T + Late Fusion + two classification heads.
* **Head 1 (BI-RADS):** Linear(1536 → 256 → 5) — primary task.
* **Head 2 (Density):** Linear(1536 → 128 → 4) — auxiliary task.
* **Imbalance Handling:**
  * `CrossEntropyLoss` with **class weights** (inverse frequency) for both tasks.
  * **WeightedRandomSampler** ensures balanced class exposure per batch.
* **Optimization:** Adam, lr = `0.0001`. Total loss = `1.0 × loss_birads + 0.5 × loss_density`.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import WeightedRandomSampler, random_split, DataLoader

print("--- 3. Feature Extraction & 4. Model Building (Dual-View Swin-T) ---")

# --- Safety Check ---
if 'df_pairs' not in globals() or 'birads_weight_tensor' not in globals():
    raise ValueError("⚠️ Please Re-Run Step 2 (Preprocessing) first!")

# --- 3.1 Data Loaders with WeightedRandomSampler ---
print("⚙️ Setting up Dual-View Data Loaders...")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),       # Safe augmentation for mammography
    transforms.ColorJitter(brightness=0.2),  # Simulate exposure variation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# IMPORTANT: Do NOT mutate random_split's internal .dataset; it can desync indices and cause OOB errors.
# Instead: split indices, then build independent datasets from filtered dataframes.
dataset_for_split = DualViewMammoDataset(df_pairs, transform=transform)
train_len = int(0.8 * len(dataset_for_split))
val_len   = len(dataset_for_split) - train_len
train_subset, val_subset = random_split(dataset_for_split, [train_len, val_len])

# Build standalone datasets using the same indices returned by random_split
train_df = df_pairs.iloc[train_subset.indices].reset_index(drop=True)
val_df   = df_pairs.iloc[val_subset.indices].reset_index(drop=True)

train_ds = DualViewMammoDataset(train_df, transform=transform)
val_ds   = DualViewMammoDataset(val_df,   transform=transform_val)

# --- WeightedRandomSampler for training set ---
# Assign per-sample weight based on its BI-RADS class (computed on train_df)
train_labels   = train_df['birads_label'].values
sample_weights = birads_weights[train_labels]  # shape: (N_train,)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_ds),
    replacement=True
)

# Use sampler in train_loader (shuffle=False when using sampler)
train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,   num_workers=2)
print(f"✅ Train: {len(train_ds)} pairs | Val: {len(val_ds)} pairs")
print("✅ WeightedRandomSampler applied to training loader.")

# --- 4.1 Define Dual-View Swin-T Architecture ---
class DualViewSwinModel(nn.Module):
    """
    Dual-View Swin Transformer with shared backbone weights.

    Forward:
      img_cc, img_mlo -> Swin-T (shared) -> feat_cc (B,768), feat_mlo (B,768)
      -> Concat -> (B, 1536)
      -> Head 1: BI-RADS (B, 5)
      -> Head 2: Density  (B, 4)
    """
    def __init__(self, num_birads=5, num_density=4):
        super(DualViewSwinModel, self).__init__()

        # Shared Swin-T backbone
        swin = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
        self.features = swin.features
        self.norm     = swin.norm
        self.permute  = swin.permute
        self.avgpool  = swin.avgpool
        self.flatten  = swin.flatten
        # Both CC and MLO pass through the SAME weights -> (B, 768) each

        FEAT_DIM = 768 * 2  # After concatenation: 1536

        # --- Head 1: BI-RADS (Primary Task) ---
        self.head_birads = nn.Sequential(
            nn.LayerNorm(FEAT_DIM),
            nn.Linear(FEAT_DIM, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_birads)
        )

        # --- Head 2: Breast Density (Auxiliary Task) ---
        self.head_density = nn.Sequential(
            nn.LayerNorm(FEAT_DIM),
            nn.Linear(FEAT_DIM, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_density)
        )

    def extract_features(self, x):
        """Pass one view through shared backbone -> (B, 768)"""
        x = self.features(x)
        x = self.norm(x)
        x = self.permute(x)
        x = self.avgpool(x)
        x = self.flatten(x)   # (B, 768)
        return x

    def forward(self, img_cc, img_mlo):
        feat_cc  = self.extract_features(img_cc)   # (B, 768)
        feat_mlo = self.extract_features(img_mlo)  # (B, 768)
        fused    = torch.cat([feat_cc, feat_mlo], dim=1)  # (B, 1536)

        out_birads  = self.head_birads(fused)   # (B, 5)
        out_density = self.head_density(fused)  # (B, 4)
        return out_birads, out_density

# --- 4.2 Setup Training ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Training Device: {DEVICE}")

model = DualViewSwinModel(num_birads=5, num_density=4).to(DEVICE)

# Class-Weighted Loss (moves weights to GPU)
criterion_birads  = nn.CrossEntropyLoss(
    weight=birads_weight_tensor.to(DEVICE)
)
criterion_density = nn.CrossEntropyLoss(
    weight=density_weight_tensor.to(DEVICE)
)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

LAMBDA_BIRADS  = 1.0
LAMBDA_DENSITY = 0.5

# --- 4.3 Training Loop ---
NUM_EPOCHS          = 20
train_losses        = []
birads_losses_hist  = []
density_losses_hist = []

print(f"\nSTARTING TRAINING ({NUM_EPOCHS} Epochs)...")
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = running_loss_b = running_loss_d = 0.0

    for i, (img_cc, img_mlo, lbl_b, lbl_d) in enumerate(train_loader):
        img_cc  = img_cc.to(DEVICE)
        img_mlo = img_mlo.to(DEVICE)
        lbl_b   = lbl_b.to(DEVICE)
        lbl_d   = lbl_d.to(DEVICE)

        optimizer.zero_grad()
        out_b, out_d = model(img_cc, img_mlo)

        loss_b = criterion_birads(out_b,  lbl_b)
        loss_d = criterion_density(out_d, lbl_d)
        loss   = LAMBDA_BIRADS * loss_b + LAMBDA_DENSITY * loss_d

        loss.backward()
        optimizer.step()

        running_loss   += loss.item()
        running_loss_b += loss_b.item()
        running_loss_d += loss_d.item()

        if i % 50 == 0:
            print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}] Step [{i}/{len(train_loader)}] "
                  f"Total: {loss.item():.4f} | BI-RADS: {loss_b.item():.4f} | Density: {loss_d.item():.4f}")

    avg_t = running_loss   / len(train_loader)
    avg_b = running_loss_b / len(train_loader)
    avg_d = running_loss_d / len(train_loader)
    train_losses.append(avg_t)
    birads_losses_hist.append(avg_b)
    density_losses_hist.append(avg_d)
    print(f"✅ Epoch {epoch+1} | Total: {avg_t:.4f} | BI-RADS: {avg_b:.4f} | Density: {avg_d:.4f}")

# --- 4.4 Save Model ---
torch.save(model.state_dict(), 'dualview_swin_multitask.pth')
print("\n💾 Model Saved: dualview_swin_multitask.pth")

# --- 4.5 Loss Curves ---
plt.figure(figsize=(10, 5))
plt.plot(train_losses,        marker='o', label='Total Loss')
plt.plot(birads_losses_hist,  marker='s', label='BI-RADS Loss (Task 1)')
plt.plot(density_losses_hist, marker='^', label='Density Loss (Task 2)')
plt.title('Multi-Task Training Loss — Dual-View Swin-T')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 5. Model Evaluation (5 Marks)

**Objective:** Evaluate the `DualViewSwinModel` on the unseen validation set for both tasks.

**Metrics Selected:**
* **Accuracy:** Overall per-task classification accuracy.
* **Precision, Recall, F1-Score:** Per-class metrics via `classification_report`.
* **Confusion Matrix:** Separate 5×5 (BI-RADS) and 4×4 (Density) matrices.

**Note on Imbalance Evaluation:**
Because WeightedRandomSampler was used during training, the validation set is evaluated on the **original unsampled distribution** — giving a realistic picture of real-world performance on imbalanced data.


In [ ]:
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("--- 5. Model Evaluation (Dual-View Swin-T) ---")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 5.1 RE-DEFINE MODEL CLASS ---
class DualViewSwinModel(nn.Module):
    def __init__(self, num_birads=5, num_density=4):
        super(DualViewSwinModel, self).__init__()
        swin = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
        self.features = swin.features
        self.norm     = swin.norm
        self.permute  = swin.permute
        self.avgpool  = swin.avgpool
        self.flatten  = swin.flatten
        FEAT_DIM = 768 * 2
        self.head_birads = nn.Sequential(
            nn.LayerNorm(FEAT_DIM), nn.Linear(FEAT_DIM, 256), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(256, num_birads)
        )
        self.head_density = nn.Sequential(
            nn.LayerNorm(FEAT_DIM), nn.Linear(FEAT_DIM, 128), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(128, num_density)
        )

    def extract_features(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = self.permute(x)
        x = self.avgpool(x)
        return self.flatten(x)

    def forward(self, img_cc, img_mlo):
        fused = torch.cat([self.extract_features(img_cc),
                           self.extract_features(img_mlo)], dim=1)
        return self.head_birads(fused), self.head_density(fused)

# --- 5.2 Load Model ---
if 'val_loader' not in globals():
    raise ValueError("⚠️ 'val_loader' missing. Please Re-Run Step 2!")

model = DualViewSwinModel(num_birads=5, num_density=4).to(DEVICE)
try:
    model.load_state_dict(torch.load('dualview_swin_multitask.pth', map_location=DEVICE))
    print("✅ Loaded 'dualview_swin_multitask.pth' successfully.")
except FileNotFoundError:
    print("⚠️ Saved model not found. Using current in-memory model state.")

model.eval()

# --- 5.3 Run Inference ---
y_true_b, y_pred_b = [], []
y_true_d, y_pred_d = [], []

print("Running inference on Validation Set...")
with torch.no_grad():
    for img_cc, img_mlo, lbl_b, lbl_d in val_loader:
        img_cc  = img_cc.to(DEVICE)
        img_mlo = img_mlo.to(DEVICE)
        out_b, out_d = model(img_cc, img_mlo)
        y_true_b.extend(lbl_b.numpy())
        y_pred_b.extend(torch.argmax(out_b, dim=1).cpu().numpy())
        y_true_d.extend(lbl_d.numpy())
        y_pred_d.extend(torch.argmax(out_d, dim=1).cpu().numpy())

# --- 5.4 Reports ---
birads_names  = ['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5']
density_names = ['Density A','Density B','Density C','Density D']

print("\n=== Task 1: BI-RADS Classification Report ===")
print(classification_report(y_true_b, y_pred_b, target_names=birads_names))
print(f"BI-RADS Accuracy: {accuracy_score(y_true_b, y_pred_b)*100:.2f}%")

print("\n=== Task 2: Breast Density Classification Report ===")
print(classification_report(y_true_d, y_pred_d, target_names=density_names))
print(f"Breast Density Accuracy: {accuracy_score(y_true_d, y_pred_d)*100:.2f}%")

# --- 5.5 Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(confusion_matrix(y_true_b, y_pred_b), annot=True, fmt='d',
            cmap='Reds', ax=axes[0],
            xticklabels=birads_names, yticklabels=birads_names)
axes[0].set_title('Confusion Matrix — Task 1: BI-RADS (5 Classes)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(confusion_matrix(y_true_d, y_pred_d), annot=True, fmt='d',
            cmap='Blues', ax=axes[1],
            xticklabels=density_names, yticklabels=density_names)
axes[1].set_title('Confusion Matrix — Task 2: Breast Density (4 Classes)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()


# **Step 5.5: Advanced Visualization & Explainability**

**1. Per-Class Accuracy Bar Chart:** Visual comparison of per-class accuracy for both tasks.

**2. Dual-View Grad-CAM:** Grad-CAM applied separately to CC and MLO views, displayed side-by-side — showing where the model attends in each view when making its BI-RADS prediction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import cv2
from sklearn.metrics import confusion_matrix

print("--- 5.5 Advanced Visualizations & Explainability ---")

# --- 1. Per-Class Accuracy ---
def per_class_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return cm.diagonal() / (cm.sum(axis=1) + 1e-6)

acc_per_b = per_class_accuracy(y_true_b, y_pred_b)
acc_per_d = per_class_accuracy(y_true_d, y_pred_d)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(birads_names, acc_per_b, color='salmon', edgecolor='black')
axes[0].set_title('Per-Class Accuracy — Task 1: BI-RADS')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(acc_per_b):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=10)

axes[1].bar(density_names, acc_per_d, color='steelblue', edgecolor='black')
axes[1].set_title('Per-Class Accuracy — Task 2: Breast Density')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1])
for i, v in enumerate(acc_per_d):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# --- 2. Dual-View Grad-CAM ---
def get_gradcam(model, img_cc_tensor, img_mlo_tensor, target_view='cc'):
    """
    Grad-CAM for Dual-View Swin-T.
    Hooks into the last LayerNorm of the backbone (swin.norm).
    target_view: 'cc' or 'mlo' — which view to compute CAM for.
    """
    gradients  = []
    activations = []

    target_layer = model.norm

    def fwd_hook(module, inp, out):
        activations.append(out.detach())
    def bwd_hook(module, gin, gout):
        gradients.append(gout[0].detach())

    hf = target_layer.register_forward_hook(fwd_hook)
    hb = target_layer.register_full_backward_hook(bwd_hook)

    model.zero_grad()
    cc_in  = img_cc_tensor.unsqueeze(0).to(DEVICE)
    mlo_in = img_mlo_tensor.unsqueeze(0).to(DEVICE)

    # Run full dual-view forward
    out_b, _ = model(cc_in, mlo_in)
    score = out_b[0, out_b.argmax(dim=1).item()]
    score.backward()

    hf.remove()
    hb.remove()

    # activations are collected for BOTH views (two forward passes through norm)
    # Index 0 = CC, Index 1 = MLO
    view_idx = 0 if target_view == 'cc' else 1
    if len(activations) < 2:
        view_idx = 0  # Fallback

    act  = activations[view_idx].squeeze(0)  # (H, W, C)
    grad = gradients[view_idx].squeeze(0)    # (H, W, C)

    H, W, C = act.shape
    act  = act.view(H * W, C)
    grad = grad.view(H * W, C)

    weights = grad.mean(dim=0)
    cam = (act * weights).sum(dim=1).view(H, W).cpu().numpy()
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224, 224))
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam

def unnormalize(tensor):
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)

def overlay_cam(img, cam):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap[:, :, ::-1]) / 255
    return np.clip(heatmap * 0.4 + img * 0.6, 0, 1)

print("\n--- Dual-View Grad-CAM Visualization ---")
BIRADS_LABELS = ['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5']
found = False

for img_cc, img_mlo, lbl_b, lbl_d in val_loader:
    for i in range(len(lbl_b)):
        true_birads = BIRADS_LABELS[lbl_b[i].item()]

        cam_cc  = get_gradcam(model, img_cc[i],  img_mlo[i], target_view='cc')
        cam_mlo = get_gradcam(model, img_cc[i],  img_mlo[i], target_view='mlo')

        disp_cc  = unnormalize(img_cc[i])
        disp_mlo = unnormalize(img_mlo[i])

        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        fig.suptitle(f'Dual-View Grad-CAM | True Label: {true_birads}',
                     fontsize=13, fontweight='bold')

        # CC Row
        axes[0, 0].imshow(disp_cc);  axes[0, 0].set_title('CC View (Original)');    axes[0, 0].axis('off')
        axes[0, 1].imshow(cam_cc, cmap='jet'); axes[0, 1].set_title('CC Attention Map'); axes[0, 1].axis('off')
        axes[0, 2].imshow(overlay_cam(disp_cc, cam_cc)); axes[0, 2].set_title('CC Overlay'); axes[0, 2].axis('off')

        # MLO Row
        axes[1, 0].imshow(disp_mlo); axes[1, 0].set_title('MLO View (Original)');   axes[1, 0].axis('off')
        axes[1, 1].imshow(cam_mlo, cmap='jet'); axes[1, 1].set_title('MLO Attention Map'); axes[1, 1].axis('off')
        axes[1, 2].imshow(overlay_cam(disp_mlo, cam_mlo)); axes[1, 2].set_title('MLO Overlay'); axes[1, 2].axis('off')

        plt.tight_layout()
        plt.show()
        found = True
        break
    if found:
        break


## 6. Discussion and Conclusion

### **Summary of Findings**
We developed a **Dual-View Multi-Task Swin Transformer** for simultaneous BI-RADS Assessment (5 classes) and Breast Density Classification (4 classes) using paired CC and MLO mammographic images from the VinDr-Mammo dataset.

### **Key Engineering Decisions**
* **Dual-View Input:** CC and MLO images are processed through a shared Swin-T backbone and fused via concatenation (Late Fusion), providing the model with complementary perspectives of each breast — closely mirroring clinical radiologist workflow.
* **Shared Backbone Weights:** One backbone serves both views, halving backbone parameters and acting as a regularizer that forces view-agnostic feature learning.
* **Class-Weighted Loss:** Inverse-frequency weights applied to `CrossEntropyLoss` penalize errors on rare BI-RADS classes (4 & 5) more heavily, improving minority-class sensitivity.
* **WeightedRandomSampler:** Ensures every training batch contains a balanced representation of all BI-RADS classes, preventing the model from ignoring rare but clinically critical categories.

### **Performance Analysis**
* **Task 1 — BI-RADS:** Combined weighted loss and sampler are expected to improve recall on BI-RADS 4 & 5 compared to unweighted baselines.
* **Task 2 — Breast Density:** Auxiliary task benefits from the rich dual-view features, reinforcing the shared backbone.
* **Dual-View Benefit:** Fusion of CC and MLO features provides 1536-dimensional representations, giving the classification heads significantly richer information than single-view (768-dim) models.

### **Limitations**
1. **Dataset Shrinkage:** Requiring complete CC+MLO pairs reduced the number of usable studies compared to single-view training.
2. **Memory Cost:** Processing two 224×224 images per forward pass doubles GPU memory requirements relative to single-view.
3. **Label Ambiguity:** BI-RADS grading has known inter-radiologist variability, introducing irreducible label noise.

### **Future Directions**
1. **Attention-Based Fusion:** Replace concatenation with cross-attention between CC and MLO features for more adaptive view integration.
2. **Swin-B Upgrade:** Larger backbone for higher representational capacity.
3. **Clinical Metadata Integration:** Incorporate patient age and family history as additional inputs to the classification heads.

---
**Conclusion:**
This project demonstrates that **Dual-View Multi-Task Learning with Swin Transformer** is a principled and clinically-motivated solution for mammographic assessment — advancing beyond single-view and single-task baselines through complementary view fusion, shared representation learning, and robust class imbalance handling.
